<a href="https://colab.research.google.com/github/Swarajk22/Swaraj_GenAI_Course/blob/main/Railway_AI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("Swaraj_Kharpude")

client = genai.Client(api_key=api_key)

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain Indian Railways in one simple sentence."
)

print(response.text)


Indian Railways is a massive, government-run train network that serves as the nation's lifeline, connecting millions of people and transporting goods across the entire country every day.


In [ ]:

!pip install -q gradio

import gradio as gr
from google import genai
from google.colab import userdata


api_key = userdata.get("Swaraj_Kharpude")
client = genai.Client(api_key=api_key)


tickets = {
    "IR1001": {
        "Passenger": "swaraj kharpude",
        "Train": "12951 Mumbai Rajdhani Express",
        "From": "Mumbai Central",
        "To": "New Delhi",
        "Date": "18 September 2026",
        "Coach": "B2",
        "Seat": "36",
        "Status": "Confirmed",
        "Fare": "₹2450"
    },

    "IR1002": {
        "Passenger": "Priya Patil",
        "Train": "12124 Deccan Queen",
        "From": "Mumbai",
        "To": "Pune",
        "Date": "20 September 2026",
        "Coach": "C2",
        "Seat": "45",
        "Status": "Confirmed",
        "Fare": "₹560"
    },

    "IR1003": {
        "Passenger": "Rahul Verma",
        "Train": "12267 Mumbai Central - Ahmedabad AC Duronto",
        "From": "Mumbai Central",
        "To": "Ahmedabad",
        "Date": "22 September 2026",
        "Coach": "A1",
        "Seat": "18",
        "Status": "Waiting List",
        "Fare": "₹1350"
    }
}


def authenticate(ticket_id):

    ticket_id = ticket_id.strip().upper()

    if ticket_id in tickets:
        return (
            f"✅ AUTHENTICATION SUCCESSFUL\n\n"
            f"Ticket ID: {ticket_id}\n"
            f"Passenger: {tickets[ticket_id]['Passenger']}\n\n"
            f"You can now ask questions about your ticket."
        )
    else:
        return (
            "❌ AUTHENTICATION FAILED\n\n"
            "Invalid Ticket ID. Access Denied."
        )

def chatbot(ticket_id, message, history):

    ticket_id = ticket_id.strip().upper()

    if ticket_id not in tickets:
        return "❌ Access Denied. Please enter a valid Ticket ID first."

    if not message.strip():
        return "Please enter your question."

    ticket = tickets[ticket_id]

    ticket_information = f"""
Ticket ID: {ticket_id}
Passenger: {ticket['Passenger']}
Train: {ticket['Train']}
From: {ticket['From']}
To: {ticket['To']}
Date: {ticket['Date']}
Coach: {ticket['Coach']}
Seat: {ticket['Seat']}
Status: {ticket['Status']}
Fare: {ticket['Fare']}
"""

    prompt = f"""
You are an Indian Railways Administration AI Assistant.

IMPORTANT SECURITY RULES:
1. The user is authenticated ONLY for Ticket ID {ticket_id}.
2. Answer ONLY using the authorized ticket information below.
3. Never reveal information about another Ticket ID.
4. If the user asks for another ticket's information, respond:
   "Access Denied. You are authenticated only for your current Ticket ID."
5. Do not invent railway information.
6. Give short, clear and helpful answers.

AUTHORIZED TICKET INFORMATION:
{ticket_information}

USER QUESTION:
{message}
"""

    response = client.models.generate_content(
        model="gemini-3.7-flash",
        contents=prompt
    )

    return response.text



with gr.Blocks(title="Indian Railways AI Assistant") as demo:

    gr.Markdown(
        """
        # 🚆 Indian Railways Secure GenAI Assistant
        ### 🔐 Ticket ID Based Secure Information Retrieval
        """
    )

    gr.Markdown(
        "Enter your **Ticket ID** to authenticate and access your railway information."
    )

    with gr.Row():
        ticket_box = gr.Textbox(
            label="🎫 Ticket ID",
            placeholder="Example: IR1001"
        )

        auth_button = gr.Button(
            "🔐 Authenticate",
            variant="primary"
        )

    auth_status = gr.Textbox(
        label="Authentication Status",
        interactive=False,
        lines=4
    )

    auth_button.click(
        authenticate,
        inputs=ticket_box,
        outputs=auth_status
    )

    gr.Markdown("---")

    chatbot_ui = gr.Chatbot(
        label="🤖 Railway AI Assistant",
        height=400
    )

    message_box = gr.Textbox(
        label="Ask your question",
        placeholder="Example: What is my train name?"
    )

    ask_button = gr.Button(
        "💬 Ask AI",
        variant="primary"
    )

    def ask_ai(ticket_id, message, history):
        answer = chatbot(ticket_id, message, history)

        if history is None:
            history = []

        history.append({
            "role": "user",
            "content": message
        })

        history.append({
            "role": "assistant",
            "content": answer
        })

        return history, ""

    ask_button.click(
        ask_ai,
        inputs=[ticket_box, message_box, chatbot_ui],
        outputs=[chatbot_ui, message_box]
    )

    gr.Markdown(
        """
        ---
        🔒 **Security:** Users can access information only for their authenticated Ticket ID.

        🧠 **Technology:** Google Gemini LLM + Python + Gradio

        🚆 **Project:** GenAI-Based Indian Railways Administration Architecture
        """
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7cf570f314d2715049.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
